In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 

In [2]:
df = pd.read_csv("../data/train.csv")
df_t = pd.read_csv("../data/test.csv")

In [3]:
df.head()

,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
0,0,36,694,4,4,4,3,1,4,1,...,4,3,1,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
1,1,56,651,5,5,5,5,5,4,4,...,5,4,3,0,0.0,Male,Loyal Customer,Business travel,Business,True
2,2,31,1371,3,4,2,5,3,3,3,...,4,5,3,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
3,3,10,1616,3,5,3,3,3,3,3,...,3,4,3,0,0.0,Male,Loyal Customer,Personal Travel,Eco,False
4,4,23,481,3,4,3,4,5,3,5,...,5,3,5,5,0.0,Female,disloyal Customer,Business travel,Eco,False


In [4]:
test_ids = df_t["id"].copy()

df.drop(["id"], axis=1, inplace=True)
df_t.drop(["id"], axis=1, inplace=True)

In [5]:
for data in [df, df_t]:

    # Delay on Air
    data["Delay on air"] = (
        data["Arrival Delay in Minutes"]
        - data["Departure Delay in Minutes"]
    )

    # Average of Age, Gate Location and Delay on Air
    data["Age_Gate_Delay_Avg"] = (
        data["Age"]
        + data["Gate location"]
        + data["Delay on air"]
    ) / 3

    # Average of Ease of Online Booking and Online Boarding
    data["Online_Experience_Avg"] = (
        data["Ease of Online booking"]
        + data["Online boarding"]
    ) / 2

    # Average of Food and Drink and On-board Service
    data["Food_Service_Avg"] = (
        data["Food and drink"]
        + data["On-board service"]
    ) / 2

    # Average of Baggage Handling and Checkin Service
    data["Baggage_Checkin_Avg"] = (
        data["Baggage handling"]
        + data["Checkin service"]
    ) / 2

In [6]:
drop_cols = [
    "Gender",
    "Departure/Arrival time convenient",
    "Arrival Delay in Minutes",
    "Departure Delay in Minutes"
]

df.drop(columns=drop_cols, inplace=True)
df_t.drop(columns=drop_cols, inplace=True)

In [7]:
print("Train shape:", df.shape)
print("Test shape:", df_t.shape)

print("\nTrain columns:")
print(df.columns.tolist())

print("\nTest columns:")
print(df_t.columns.tolist())

Train shape: (699635, 23)
Test shape: (299844, 22)

Train columns:
['Age', 'Flight Distance', 'Inflight wifi service', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Cleanliness', 'Customer Type', 'Type of Travel', 'Class', 'satisfaction', 'Delay on air', 'Age_Gate_Delay_Avg', 'Online_Experience_Avg', 'Food_Service_Avg', 'Baggage_Checkin_Avg']

Test columns:
['Age', 'Flight Distance', 'Inflight wifi service', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Cleanliness', 'Customer Type', 'Type of Travel', 'Class', 'Delay on air', 'Age_Gate_Delay_Avg', 'Online_Experience_Avg', 'Food_Service_Avg', 'Baggage_Checkin_Avg']


In [8]:
import joblib
from sklearn.model_selection import train_test_split

# PREPROCESSING
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PowerTransformer, FunctionTransformer 
from sklearn.impute import SimpleImputer

# MODELS
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# MODEL EVALUATION
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# HYPERPARAMETER TUNING
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_score
)

In [9]:
X = df.drop(['satisfaction'], axis = 1)
y = df['satisfaction']

In [10]:
num_cols = X.select_dtypes(include = ['int', 'float']).columns
cat_cols = X.select_dtypes(include='object').columns

In [11]:
left_skewed = []
right_skewed = []
for cols in num_cols:
    if df[cols].skew() > 0.5:
        print(f"{cols} is skewed with skewness: {df[cols].skew()}")
        right_skewed.append(cols)

    elif df[cols].skew() < -0.5:
        print(f"{cols} is negatively skewed with skewness: {df[cols].skew()}")
        left_skewed.append(cols)


print(f"Left skewed features: {left_skewed}")
print(f"Right skewed features: {right_skewed}")

Flight Distance is skewed with skewness: 1.0804656071734196
Seat comfort is negatively skewed with skewness: -0.6161920579994616
On-board service is negatively skewed with skewness: -0.5593974280113893
Baggage handling is negatively skewed with skewness: -0.8388565290421319
Delay on air is skewed with skewness: 2.10599987694359
Baggage_Checkin_Avg is negatively skewed with skewness: -0.5408097283204384
Left skewed features: ['Seat comfort', 'On-board service', 'Baggage handling', 'Baggage_Checkin_Avg']
Right skewed features: ['Flight Distance', 'Delay on air']
